# BD-KDD Kidney Disease — ANOVA + DODA Feature Selection

## Experimental objective

Evaluate conventional **ANOVA feature selection** against **ANOVA + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

ANOVA scores **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if ANOVA selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [14]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


PROCESSED BD-KDD DATASET
Rows    : 988
Columns : 25


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane,Class
0,58,147,1.025,1,3,0,1,0,0,177,...,43,12342,4.3,1,0,0,1,1,0,0
1,71,142,1.025,2,3,1,0,1,0,247,...,47,7249,3.7,0,1,1,0,1,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,37,4111,3.7,0,0,0,0,0,1,0
3,34,70,1.005,1,3,1,1,0,0,215,...,44,4682,4.0,1,0,1,1,0,1,1
4,62,120,1.020,4,0,1,0,1,0,143,...,54,5161,4.7,0,0,1,0,1,0,1


In [15]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


FEATURE / TARGET SEPARATION
Predictors (X) : 24
Records        : 988
Target (y)     : Class
Target values  : [0, 1]


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Hemo,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane
0,58,147,1.025,1,3,0,1,0,0,177,...,9.3,43,12342,4.3,1,0,0,1,1,0
1,71,142,1.025,2,3,1,0,1,0,247,...,16.9,47,7249,3.7,0,1,1,0,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,16.6,37,4111,3.7,0,0,0,0,0,1
3,34,70,1.005,1,3,1,1,0,0,215,...,15.4,44,4682,4.0,1,0,1,1,0,1
4,62,120,1.020,4,0,1,0,1,0,143,...,13.9,54,5161,4.7,0,0,1,0,1,0


0    0
1    1
2    0
3    1
4    1
Name: Class, dtype: int64

In [16]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


,Count,Percentage
Class,,
0,481,48.68
1,507,51.32


In [17]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


DATA TYPES


,Data Type
Age,int64
Bp,int64
Sg,float64
Al,int64
Su,int64
Rbc,int64
Pc,int64
Pcc,int64
Ba,int64
Bgr,int64


MISSING VALUES


,Missing Count,Missing Percentage (%)
Age,0,0.0
Bp,0,0.0
Sg,0,0.0
Al,0,0.0
Su,0,0.0
Rbc,0,0.0
Pc,0,0.0
Pcc,0,0.0
Ba,0,0.0
Bgr,0,0.0


No missing values detected. No imputation will be performed.


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [18]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAIN-TEST SPLIT
X_train : (790, 24)
X_test  : (198, 24)
y_train : (790,)
y_test  : (198,)

Training class distribution:


Class
1    51.27
0    48.73
Name: proportion, dtype: float64


Testing class distribution:


Class
1    51.52
0    48.48
Name: proportion, dtype: float64

In [19]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# ANOVA F-statistics are scale-invariant, so feature selection is performed
# on the original training-fold values.
#
# Scaling is applied later only for Logistic Regression.

print("ANOVA will be fitted on the original training data.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


ANOVA will be fitted on the original training data.
Logistic Regression will be scaled after feature selection.
Random Forest and XGBoost will use the selected features without scaling.


# 1. ANOVA Baseline

ANOVA is used as the statistical feature-selection baseline.

**Important:** ANOVA scores all 24 predictors first. Top-K is applied only after the complete ranking has been generated.


In [20]:
# =============================================================================
# STEP 7: ANOVA SCORES FOR ALL FEATURES
# =============================================================================

from sklearn.feature_selection import SelectKBest, f_classif

anova_selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

anova_selector.fit(X_train, y_train)

anova_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "ANOVA_F_Score": anova_selector.scores_,
    "ANOVA_P_Value": anova_selector.pvalues_
}).sort_values(
    "ANOVA_F_Score",
    ascending=False
).reset_index(drop=True)

anova_scores["ANOVA_Rank"] = np.arange(1, len(anova_scores) + 1)

display(anova_scores)


,Feature,ANOVA_F_Score,ANOVA_P_Value,ANOVA_Rank
0,Bp,8.902625,0.002936,1
1,Su,1.475878,0.224784,2
2,Ba,1.387509,0.239182,3
3,Bu,1.295059,0.255464,4
4,Ane,1.236786,0.266431,5
5,Rbcc,0.835335,0.361014,6
6,Pcv,0.779051,0.377701,7
7,Sc,0.770546,0.380316,8
8,Appet,0.634996,0.425769,9
9,Bgr,0.596137,0.440287,10


In [21]:
# =============================================================================
# STEP 8: TOP-K ANOVA FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

anova_results = {}

for top_k in TOP_K_VALUES:

    selected = anova_scores.head(top_k)["Feature"].tolist()

    anova_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"ANOVA TOP-{top_k}")
    print("=" * 70)
    print(selected)


Total predictors: 24
Top-K values: [5, 10, 15, 20]
ANOVA TOP-5
['Bp', 'Su', 'Ba', 'Bu', 'Ane']
ANOVA TOP-10
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr']
ANOVA TOP-15
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr', 'Sg', 'Dm', 'Pcc', 'Pe', 'Al']
ANOVA TOP-20
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr', 'Sg', 'Dm', 'Pcc', 'Pe', 'Al', 'Pot', 'Wbcc', 'Cad', 'Rbc', 'Htn']


In [22]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


['Logistic Regression', 'Random Forest', 'XGBoost']


In [23]:
# =============================================================================
# STEP 10: ANOVA BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_anova_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = anova_results[top_k]["X_train"]
    Xte_selected = anova_results[top_k]["X_test"]
    selected_features = anova_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_anova_rows.append({
            "Method": "ANOVA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_anova_results_df = pd.DataFrame(fixed_anova_rows)
display(fixed_anova_results_df.round(4))


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# STEP 11: SAVE ANOVA BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "anova_baseline_results.csv"
fixed_anova_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


Saved: C:\Users\johnm\msc_research\results\kidney_disease\baseline\anova_baseline_results.csv


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
ANOVA scores all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA rank fusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated ANOVA Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [ ]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)


DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py


In [24]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

provider = JSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the JSON file directly.
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    weights_payload = json.load(f)

clinical_weights = weights_payload["weights"]

missing_weights = sorted(set(X.columns) - set(clinical_weights))
extra_weights = sorted(set(clinical_weights) - set(X.columns))

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights


Using clinical weights: C:\Users\johnm\msc_research\config\clinical_weights\BD_KDD_clinical_weights.json
Clinical weights: 24
Predictors:       24
Missing weights:  []
Extra weights:    []


In [25]:
# =============================================================================
# STEP 14: DODA — SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

rank_doda_results = {}

for top_k in TOP_K_VALUES:

    anova_operator = SklearnAdapter(
        SelectKBest(
            score_func=f_classif,
            k="all"     # CRITICAL: all 24 features reach DODA
        )
    )

    doda_selector = DODASelector(
        operators=[anova_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit only on training data.
    doda_selector.fit(
        X_train,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000002B6150138C0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.015848009107691952), 'Bp': np.float64(8.902625130307502), 'Sg': np.float64(0.5823525344971223), 'Al': np.float64(0.16292862378792028), 'Su': np.float64(1.4758784262753983), 'Rbc': np.float64(0.08822043465353854), 'Pc': np.float64(8.098897814608765e-05), 'Pcc': np.float64(0.4320562634767095), 'Ba': np.float64(1.387509378150789), 'Bgr': np.float64(0.5961366416517426), 'Bu': np.float64(1.2950594931817596), 'Sc': np.float64(0.7705460731720983), 'Sod': np.float64(0.0005941532857815975), 'Pot': np.float64(0.14517187640480858), 'Hemo': np.float64(0.05114806428390342), 'Pcv': np.float64(0.7790510482306884), 'Wbcc': np.float64(0.10643069394407427), 'Rbcc':

In [26]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


DODA RANK COMPARISON — TOP-10

MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      Bp          1              2           1            0
1      Su          2              5           2            0
2      Ba          3              9           3            0
3      Bu          4             11           4            0
4     Ane          5             24           5            0
5    Rbcc          6             18           6            0
6     Pcv          7             16           7            0
7      Sc          8             12           8            0
8   Appet          9             22           9            0
9     Bgr         10             10          10            0


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,Bp,1,1.0000,2,1.0,1,0.0328,0
1,Su,2,0.1658,5,1.0,2,0.0325,0
2,Ba,3,0.1559,9,1.0,3,0.0323,0
3,Bu,4,0.1455,11,1.0,4,0.0320,0
4,Ane,5,0.1389,24,1.0,5,0.0318,0
5,Rbcc,6,0.0938,18,1.0,6,0.0315,0
6,Pcv,7,0.0875,16,1.0,7,0.0313,0
7,Sc,8,0.0866,12,1.0,8,0.0311,0
8,Appet,9,0.0713,22,1.0,9,0.0309,0
9,Bgr,10,0.0670,10,1.0,10,0.0307,0



Selected features:
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr']


In [27]:
# =============================================================================
# STEP 15B: EXPLICIT ANOVA vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical ANOVA rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: ANOVA rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# ANOVA / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "ANOVA_Normalized_Score"
    ]
)

math_rank["ANOVA_Rank"] = (
    math_rank["ANOVA_Normalized_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "ANOVA_Normalized_Score",
            "ANOVA_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["ANOVA_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"ANOVA vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

ANOVA vs DODA RANK-SHIFT ANALYSIS — TOP-10


,Feature,ANOVA_Normalized_Score,ANOVA_Rank,DODA_Final_Score,DODA_Rank,Clinical_Weight,Rank_Change
0,Bp,1.000000,1,0.032787,1,1.0,0
1,Su,0.165780,2,0.032522,2,1.0,0
2,Ba,0.155854,3,0.032266,3,1.0,0
3,Bu,0.145469,4,0.032018,4,1.0,0
4,Ane,0.138924,5,0.031778,5,1.0,0
5,Rbcc,0.093830,6,0.031545,6,1.0,0
6,Pcv,0.087508,7,0.031319,7,1.0,0
7,Sc,0.086553,8,0.031099,8,1.0,0
8,Appet,0.071327,9,0.030886,9,1.0,0
9,Bgr,0.066962,10,0.030679,10,1.0,0


In [28]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


,Feature,Clinical_Weight
0,Age,1.0
1,Bp,1.0
2,Sg,1.0
3,Al,1.0
4,Su,1.0
5,Rbc,1.0
6,Pc,1.0
7,Pcc,1.0
8,Ba,1.0
9,Bgr,1.0


In [29]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,DODA,5,Logistic Regression,"Bp, Su, Ba, Bu, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,DODA,5,Random Forest,"Bp, Su, Ba, Bu, Ane",0.5303,0.5474,0.5098,0.5279,0.5600
2,DODA,5,XGBoost,"Bp, Su, Ba, Bu, Ane",0.5152,0.5288,0.5392,0.5340,0.5198
3,DODA,10,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5606,0.5630,0.6569,0.6063,0.5895
4,DODA,10,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5051,0.5208,0.4902,0.5051,0.5129
5,DODA,10,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.4798,0.4947,0.4608,0.4772,0.4959
6,DODA,15,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5660,0.5882,0.5769,0.5684
7,DODA,15,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5686,0.5686,0.5686,0.5510
8,DODA,15,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5354,0.5500,0.5392,0.5446,0.5448
9,DODA,20,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5505,0.5596,0.5980,0.5782,0.5751


In [30]:
# =============================================================================
# STEP 18: FIXED-SPLIT ANOVA vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_anova_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,ANOVA,5,Logistic Regression,"Bp, Su, Ba, Bu, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,ANOVA,5,Random Forest,"Bp, Su, Ba, Bu, Ane",0.5303,0.5474,0.5098,0.5279,0.5600
2,ANOVA,5,XGBoost,"Bp, Su, Ba, Bu, Ane",0.5152,0.5288,0.5392,0.5340,0.5198
3,ANOVA,10,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5606,0.5630,0.6569,0.6063,0.5895
4,ANOVA,10,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5051,0.5208,0.4902,0.5051,0.5129
5,ANOVA,10,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.4798,0.4947,0.4608,0.4772,0.4959
6,ANOVA,15,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5660,0.5882,0.5769,0.5684
7,ANOVA,15,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5686,0.5686,0.5686,0.5510
8,ANOVA,15,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5354,0.5500,0.5392,0.5446,0.5448
9,ANOVA,20,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5505,0.5596,0.5980,0.5782,0.5751


In [31]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    anova_set = set(anova_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = anova_set & doda_set
    union = anova_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "ANOVA_Features": ", ".join(anova_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": anova_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


,Top_K,ANOVA_Features,DODA_Features,Same_Feature_Set,Overlap_Count,Jaccard,Features_Changed
0,5,"Bp, Su, Ba, Bu, Ane","Bp, Su, Ba, Bu, Ane",True,5,1.0,0
1,10,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr","Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",True,10,1.0,0
2,15,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...","Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",True,15,1.0,0
3,20,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...","Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",True,20,1.0,0


In [32]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "anova_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "anova_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


Saved results to: C:\Users\johnm\msc_research\results\kidney_disease\doda


# 3. Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing ANOVA or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [33]:
# =============================================================================
# STEP 21: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


Top-K values: [5, 10, 15, 20]
CV runs: 25


In [34]:
# =============================================================================
# STEP 22: 5×5 REPEATED CV — ANOVA vs DODA
# =============================================================================

cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    # -----------------------------------------------------------------
    # ANOVA scores ALL features inside this training fold
    # -----------------------------------------------------------------

    anova_all = SelectKBest(
        score_func=f_classif,
        k="all"
    )

    anova_all.fit(
        X_train_fold,
        y_train_fold
    )

    anova_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": anova_all.scores_
    }).sort_values(
        "Score",
        ascending=False
    )

    # -----------------------------------------------------------------
    # DODA also receives ALL features and performs final Top-K
    # -----------------------------------------------------------------

    doda_operator = SklearnAdapter(
        SelectKBest(
            score_func=f_classif,
            k="all"
        )
    )

    fold_provider = JSONProvider(str(WEIGHTS_PATH))

    doda_selector = DODASelector(
        operators=[doda_operator],
        provider=fold_provider,
        fusion=RankFusion(),
        top_k=max(TOP_K_VALUES)
    )

    # Fit once at max K, then obtain rankings.
    doda_selector.fit(
        X_train_fold,
        y_train_fold
    )

    # -------------------------------------------------------------
    # We need the selector independently for each Top-K.
    # Refit because top_k is part of selector configuration.
    # -------------------------------------------------------------

    for top_k in TOP_K_VALUES:

        # -------------------------
        # ANOVA Top-K
        # -------------------------

        anova_features = (
            anova_fold_scores
            .head(top_k)["Feature"]
            .tolist()
        )

        # -------------------------
        # DODA Top-K
        # -------------------------

        fold_doda_operator = SklearnAdapter(
            SelectKBest(
                score_func=f_classif,
                k="all"
            )
        )

        fold_doda = DODASelector(
            operators=[fold_doda_operator],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        # -------------------------
        # Model loop
        # -------------------------

        feature_sets = {
            "ANOVA": anova_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "Recall": recall_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "F1": f1_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "ROC_AUC": roc_auc_score(
                        y_val_fold, y_prob
                    ),
                    "Selected_Features": ", ".join(selected_features)
                })

cv_results_df = pd.DataFrame(cv_results)

print("Completed.")
print(f"Rows generated: {len(cv_results_df)}")

assert len(cv_results_df) == 25 * 4 * 2 * 3

display(cv_results_df.head())


Run 01/25
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000002B61ADA2850>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.5692921565173608), 'Bp': np.float64(8.782731827841262), 'Sg': np.float64(3.670322367176968), 'Al': np.float64(0.0457132428955518), 'Su': np.float64(1.7944164384023895), 'Rbc': np.float64(0.3016609605897061), 'Pc': np.float64(0.3327267055767738), 'Pcc': np.float64(0.013120250838075088), 'Ba': np.float64(1.0578743890740132), 'Bgr': np.float64(0.6151995671004556), 'Bu': np.float64(8.565899281370292e-05), 'Sc': np.float64(0.8746376579674088), 'Sod': np.float64(0.6491999804114769), 'Pot': np.float64(0.9935838942402748), 'Hemo': np.float64(0.5374636482028966), 'Pcv': np.float64(0.11476801752744284), 'Wbcc': np.float64(1.0518558894081236), 'Rbc

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# STEP 23: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


In [ ]:
# =============================================================================
# STEP 24: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


# 4. Feature-set Stability and Rank/Set Change

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [ ]:
# =============================================================================
# STEP 25: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        anova_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        anova_set = set(
            anova_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(anova_set & doda_set)
        union = len(anova_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": anova_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


In [ ]:
# =============================================================================
# STEP 26: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["ANOVA", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


In [ ]:
# =============================================================================
# STEP 27: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        anova = cv_results_df[
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(anova) == 25
        assert len(doda) == 25
        assert (anova["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            anova_values = anova[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - anova_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    anova_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "ANOVA_Mean": np.mean(anova_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_ANOVA": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


In [ ]:
# =============================================================================
# STEP 28: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "anova_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")


In [ ]:
# =============================================================================
# STEP 29: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"CV result rows: {len(cv_results_df)}")
print(f"Expected CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)

for top_k in TOP_K_VALUES:
    assert len(anova_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


# Methodological corrections made in this final version

### 1. Correct predictor count
BD-KDD has **25 processed columns**, but one is the binary target `Class`. Therefore the feature matrix contains **24 predictors**.

### 2. Top-K values
The experiment uses:

**Top-5, Top-10, Top-15, Top-20**

This provides approximately 20%, 40%, 60%, and 80% feature budgets over the 24 predictors. Top-24 is deliberately not used as a selection condition because retaining every feature cannot demonstrate feature-selection behavior.

### 3. Removed imputation
BD-KDD has zero missing values, so no imputation is performed.

### 4. Correct DODA architecture
ANOVA is configured with `k="all"` inside DODA. DODA receives all 24 statistical scores and performs the final Top-K selection.

### 5. No data leakage in CV
ANOVA and DODA are refit inside every training fold. Validation-fold data is never used to calculate feature-selection scores.

### 6. Model-specific scaling
Scaling is applied only to Logistic Regression and is fitted on the training data/fold. Random Forest and XGBoost use the selected features without scaling.

### 7. Consistent result paths
All outputs are stored under:

```text
results/
└── kidney_disease/
    ├── baseline/
    ├── doda/
    ├── cv/
    └── statistics/
```

### 8. Fixed split and repeated CV are separated
The fixed 80/20 experiment is an initial reproducible comparison. The 5×5 repeated stratified CV is the robustness analysis.

### 9. Feature-set change and rank change are separated
Jaccard measures feature-set overlap. DODA's rank-comparison output is retained separately because a feature can change rank without entering or leaving Top-K.

### 10. Clinical weights remain external
The clinical weights are loaded from the pre-specified JSON and are not derived from ANOVA scores, target correlations, model performance, or test-fold information.


## Interpretation rule

Do not assume that DODA must improve predictive performance.

The experiment should answer three separate questions:

1. **Does DODA change feature prioritization?**
2. **How stable are ANOVA and DODA selections across repeated folds?**
3. **What happens to predictive performance when the feature priorities are changed?**

If DODA changes the selected features while predictive performance remains similar, that is a different finding from DODA improving predictive performance.

All conclusions should be based on the generated results rather than predetermined expectations.
